# FFAI Text Chunking Strategies

This notebook demonstrates the chunking strategies available in FFAI's RAG module,
located at `src/rag/splitters/`. Each strategy splits text into `TextChunk` objects
with content, position info, and metadata.

Strategies covered:

1. **Character** -- fixed-size windows with word-boundary awareness
2. **Recursive** -- hierarchical separator-based splitting
3. **Markdown** -- header-aware section splitting
4. **Code** -- AST-pattern-based function/class splitting
5. **Hierarchical** -- parent-child chunk relationships

In [ ]:
# Walk up from CWD to find project root (where pyproject.toml lives)
import sys
from pathlib import Path

_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / 'pyproject.toml').is_file():
        _project_root = _p
        break

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from src.rag.splitters import (  # noqa: E402
    CharacterChunker,
    CodeChunker,
    HierarchicalChunker,
    MarkdownChunker,
    RecursiveChunker,
    chunk_text,
    get_chunker,
    list_chunkers,
)

print('Imports OK')

In [ ]:
# Download sample documents (Alice in Wonderland from Gutenberg + bundled tutorial)
from examples._rag_data.download import download

docs = download()
print(f'\nDocuments available: {list(docs.keys())}')

In [ ]:
# Load the sample texts as strings
alice_text = (docs['alice_wonderland.txt']).read_text(encoding='utf-8')
tutorial_text = (docs['python_tutorial.md']).read_text(encoding='utf-8')

print(f'alice_wonderland.txt: {len(alice_text):,} chars')
print(f'python_tutorial.md:   {len(tutorial_text):,} chars')

<div class="page-break"></div>

---

## Available Strategies

`list_chunkers()` returns all registered strategy names. Each name can be
passed to `get_chunker()` or used as the `strategy` argument to `chunk_text()`.

In [ ]:
strategies = list_chunkers()
print('Available chunking strategies:')
for s in strategies:
    print(f'  - {s}')

<div class="page-break"></div>

---

## Character Chunking

`CharacterChunker` splits text into fixed-size windows, optionally respecting
word boundaries. It is the simplest strategy and works well for plain prose.

Parameters used: `chunk_size=200`, `chunk_overlap=50`.

In [ ]:
# Chunk the first 2000 chars of Alice in Wonderland
char_chunker = CharacterChunker(chunk_size=200, chunk_overlap=50)
char_chunks = char_chunker.chunk(alice_text[:2000])

print(f'CharacterChunker produced {len(char_chunks)} chunks\n')
for c in char_chunks[:3]:
    preview = c.content[:80].replace('\n', '\\n')
    print(f'Chunk {c.chunk_index}: [{c.start_char}:{c.end_char}] ({len(c.content)} chars)')
    print(f'  "{preview}..."')
    print()

<div class="page-break"></div>

---

## Recursive Chunking

`RecursiveChunker` splits text hierarchically using a list of separators
(paragraphs, then lines, then sentences, then words). It produces more
semantically coherent chunks than character-based splitting.

Parameters used: `chunk_size=500`, `chunk_overlap=100`.

In [ ]:
# Chunk the Python tutorial with recursive strategy
rec_chunker = RecursiveChunker(chunk_size=500, chunk_overlap=100)
rec_chunks = rec_chunker.chunk(tutorial_text)

print(f'RecursiveChunker produced {len(rec_chunks)} chunks\n')
print('Separator hierarchy: paragraphs -> lines -> sentences -> words\n')

for c in rec_chunks:
    has_header = c.metadata.get('header') is not None if c.metadata else False
    header_info = f' [header={c.metadata["header"]}]' if has_header else ''
    preview = c.content[:80].replace('\n', '\\n')
    print(f'Chunk {c.chunk_index}: [{c.start_char}:{c.end_char}] ({len(c.content)} chars){header_info}')
    print(f'  "{preview}..."')
    print()

Recursive chunking does not track document structure (like headers) in metadata.
For structure-aware splitting, use the **Markdown** chunker (next section).

The separator hierarchy keeps paragraphs together when they fit within
`chunk_size`, then falls back to sentences, then words.

<div class="page-break"></div>

---

## Markdown Chunking

`MarkdownChunker` splits by headers (`h1`, `h2`, `h3`, ...) and preserves
header information in each chunk's `metadata` dict under the `header` and
`header_level` keys.

Parameters used: `chunk_size=500`, `split_headers=["h1", "h2", "h3"]`.

In [ ]:
# Chunk the Python tutorial with markdown-aware strategy
md_chunker = MarkdownChunker(chunk_size=500, split_headers=['h1', 'h2', 'h3'])
md_chunks = md_chunker.chunk(tutorial_text)

print(f'MarkdownChunker produced {len(md_chunks)} chunks\n')

for c in md_chunks:
    header = c.metadata.get('header', '(no header)') if c.metadata else '(no metadata)'
    level = c.metadata.get('header_level', '-') if c.metadata else '-'
    preview = c.content[:80].replace('\n', '\\n')
    print(f'Chunk {c.chunk_index}: [{c.start_char}:{c.end_char}] ({len(c.content)} chars)')
    print(f'  Header: {header}  (level={level})')
    print(f'  "{preview}..."')
    print()

<div class="page-break"></div>

---

## Code Chunking

`CodeChunker` splits source code by structural boundaries (functions, classes)
using language-specific regex patterns. It records `block_type` and `block_name`
in each chunk's metadata.

Parameters used: `language="python"`, `split_by="function"`.

In [ ]:
# Sample Python module with functions and a class
sample_code = '''\
import os
import json


def load_config(path):
    with open(path) as f:
        return json.load(f)


def save_config(path, config):
    with open(path, 'w') as f:
        json.dump(config, f, indent=2)


def merge_configs(base, override):
    merged = base.copy()
    merged.update(override)
    return merged


class ConfigManager:
    def __init__(self, config_dir):
        self.config_dir = config_dir
        self._cache = {}

    def get(self, name):
        if name not in self._cache:
            path = os.path.join(self.config_dir, f"{name}.json")
            self._cache[name] = load_config(path)
        return self._cache[name]
'''

print(f'Sample code: {len(sample_code)} chars')
print(sample_code[:200] + '...')

In [ ]:
# Chunk the sample code by functions
code_chunker = CodeChunker(language='python', split_by='function')
code_chunks = code_chunker.chunk(sample_code)

print(f'CodeChunker produced {len(code_chunks)} chunks\n')

for c in code_chunks:
    block_type = c.metadata.get('block_type', '?') if c.metadata else '?'
    block_name = c.metadata.get('block_name', '?') if c.metadata else '?'
    print(f'Chunk {c.chunk_index}: [{c.start_char}:{c.end_char}] ({len(c.content)} chars)')
    print(f'  block_type={block_type}  block_name={block_name}')
    preview = c.content[:100].replace('\n', '\\n')
    print(f'  "{preview}"')
    print()

<div class="page-break"></div>

---

## Hierarchical Chunking

`HierarchicalChunker` creates a two-level tree of chunks:

- **Parent chunks** (large) provide broad context
- **Child chunks** (small) are used for precise retrieval
- Each child stores `parent_id`; each parent stores `child_ids`

Parameters used: `chunk_size=200`, `parent_chunk_size=800`.

In [ ]:
# Chunk first 3000 chars of Alice with hierarchical strategy
hier_chunker = HierarchicalChunker(chunk_size=200, parent_chunk_size=800)
hier_chunks = hier_chunker.chunk(alice_text[:3000])

parents = [c for c in hier_chunks if c.hierarchy_level == 0]
children = [c for c in hier_chunks if c.hierarchy_level > 0]

print(f'Total chunks: {len(hier_chunks)} ({len(parents)} parents, {len(children)} children)\n')

In [ ]:
# Print parent chunks with their child relationships
print('=== Parent Chunks ===\n')
for p in parents:
    preview = p.content[:80].replace('\n', '\\n')
    print(f'Parent {p.id[:8]}... [{p.start_char}:{p.end_char}] ({len(p.content)} chars)')
    child_summary = ', '.join(cid[:8] + '...' for cid in p.child_ids)
    print(f'  child_ids: [{child_summary}]')
    print(f'  "{preview}..."')
    print()

In [ ]:
# Print child chunks with parent back-links
print('=== Child Chunks (first 6) ===\n')
for c in children[:6]:
    preview = c.content[:80].replace('\n', '\\n')
    pid = c.parent_id[:8] + '...' if c.parent_id else 'None'
    print(f'Child {c.id[:8]}... parent={pid} [{c.start_char}:{c.end_char}] ({len(c.content)} chars)')
    print(f'  "{preview}..."')
    print()

Each child's `parent_id` points to the parent chunk that contains it.
At retrieval time, you can fetch a child's parent for additional context.

<div class="page-break"></div>

---

## Factory and Convenience Functions

Instead of importing and constructing chunker classes directly, you can use:

- `get_chunker(strategy)` -- returns a configured `ChunkerBase` instance
- `chunk_text(text, strategy)` -- one-liner that creates a chunker and chunks

In [ ]:
# Factory: get_chunker() returns a configured chunker instance
factory_md = get_chunker('markdown', chunk_size=500, split_headers=['h1', 'h2'])
factory_chunks = factory_md.chunk(tutorial_text)
print(f'get_chunker("markdown") -> {type(factory_md).__name__}')
print(f'Produced {len(factory_chunks)} chunks\n')

# Convenience: chunk_text() is a one-liner
quick_chunks = chunk_text(alice_text[:1000], strategy='recursive', chunk_size=300)
print(f'chunk_text(strategy="recursive") -> {len(quick_chunks)} chunks')
for c in quick_chunks:
    print(f'  Chunk {c.chunk_index}: {len(c.content)} chars [{c.start_char}:{c.end_char}]')

<div class="page-break"></div>

---

## Summary

In [ ]:
print(f'{"Strategy":<15} {"Typical Use Case":<40} {"Key Parameters"}')
print('-' * 90)
rows = [
    ('character', 'Plain text, simple splitting', 'chunk_size, chunk_overlap, respect_word_boundaries'),
    ('recursive', 'General-purpose prose, keeps paragraphs together', 'chunk_size, chunk_overlap, separators'),
    ('markdown', 'Markdown docs with header structure', 'chunk_size, split_headers, preserve_structure'),
    ('code', 'Source code splitting by functions/classes', 'language, split_by'),
    ('hierarchical', 'Parent-child context for retrieval', 'chunk_size, parent_chunk_size, max_levels'),
]
for name, use_case, params in rows:
    print(f'{name:<15} {use_case:<40} {params}')